# Day 2: train baseline and redacted-data models

This notebook uses GPT-2, matching the reference project, to train two small experimental versions: one on the original fictional conversations and one on the same conversations with private values replaced by `[REDACTED]`. Training both with the same model, seed, and settings gives us a fairer comparison.

The first run is a **16-example smoke test**, not a result. It checks that packages, model download, device, and save paths work. Only run the full 1,000-example training after that succeeds. On this 8 GB laptop, use free Colab with a GPU if available; a local CPU run may be slow. The notebook does not start a full training run automatically.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data" / "synthetic" / "training_conversations.jsonl").exists(), (
    "Run notebook 01 or generate the Day 1 dataset first."
)
print(f"Project root: {PROJECT_ROOT}")
print("Install dependencies in the next cell if needed.")

In [ ]:
# Uncomment this in Colab or a fresh environment to install project dependencies.
# %pip install -r requirements.txt

## Smoke test first

This trains both versions on 16 examples for one epoch and saves them under `models/day2-smoke/`. The public GPT-2 weights must be downloaded once. No generated resident data is sent anywhere by this code.

In [ ]:
import subprocess

SMOKE_TEST = True
command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "train_models.py"),
    "--data-dir", str(PROJECT_ROOT / "data" / "synthetic"),
    "--model-name", "gpt2",
    "--output-dir", str(PROJECT_ROOT / "models" / ("day2-smoke" if SMOKE_TEST else "day2-full")),
    "--seed", "42",
    "--epochs", "1",
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "128",
]
if SMOKE_TEST:
    command.extend(["--max-train-examples", "16"])
else:
    print("Full run: 1,000 training examples per model. This may take a while.")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## Full Day 2 run

After the smoke test finishes and both saved model folders exist, set `SMOKE_TEST = False` in the previous cell and run it again. The full run uses 1,000 training examples for each model, one epoch, batch size 1, gradient accumulation 8, and sequence length 128. These are deliberately modest starting settings; record any changes you make. The saved `run_config.json` files capture the settings and device used.

Do not use the test records to train or tune either model. We will use the held-out test data and the privacy probes in the evaluation stage.